# Crash-to-Contact — Bronze / Silver Exploratory Data Analysis

Read-only exploration of everything currently on disk under `data/bronze/` and
`data/silver/`. This notebook does not import anything from `src/transform` or
`src/ingest` — it queries the parquet files directly with DuckDB, the same way
a reviewer or a downstream analyst would, so it stays honest about what the
pipeline actually persisted rather than what the code intends.

Sections:

1. Setup
2. Bronze — watermark store & manifest
3. Bronze — Montgomery County (`bhju-22kf` / `mmzv-x632` / `n7fk-dce5`)
4. Bronze — TxDOT CRIS (`cris_crash`)
5. Bronze — FARS (annual zips, 2019–2024)
6. Silver — per-source current/history tables
7. Silver — `crash_current` (the unified, cross-source, non-entity-resolved grain)
8. Visual EDA
9. Live defect reproduction (cross-check against `DATA_QUALITY.md` claims)
10. Scratch space

Re-run `python -m src.transform.build` first if `data/silver/` looks stale
relative to `data/bronze/`.


## 1. Setup

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

from src.config import BRONZE_DIR, SILVER_DIR  # noqa: E402

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 80)

con = duckdb.connect()
con.execute("SET threads TO 4")

print("bronze:", BRONZE_DIR)
print("silver:", SILVER_DIR)


bronze: /Users/dev/Documents/ravlco-data-engineering-assignment/data/bronze
silver: /Users/dev/Documents/ravlco-data-engineering-assignment/data/silver


In [2]:
def partitions(base: Path) -> list[Path]:
    """Timestamped partition dirs under `base`, oldest first (names sort lexically)."""
    if not base.exists():
        return []
    return sorted(p for p in base.iterdir() if p.is_dir())


def latest_partition(base: Path) -> Path | None:
    parts = partitions(base)
    return parts[-1] if parts else None


def q(sql: str) -> pd.DataFrame:
    """Run a query against `con` and return a DataFrame."""
    return con.sql(sql).df()


def schema(glob: str) -> pd.DataFrame:
    return q(f"DESCRIBE SELECT * FROM read_parquet('{glob}', union_by_name=True)")


MONTGOMERY_DATASETS = {
    "bhju-22kf": "incidents (crash grain)",
    "mmzv-x632": "drivers (party grain, denormalized crash attrs)",
    "n7fk-dce5": "non_motorists (party grain)",
}


## 2. Bronze — watermark store & manifest

`data/bronze/_watermarks.duckdb` is the durable cursor store: `watermarks` (current position per source), `watermark_history` (append-only advance log), `bronze_manifest` (one row per raw artifact — path, sha256, bytes, row count, upstream Last-Modified/ETag).

In [3]:
wm_path = BRONZE_DIR / "_watermarks.duckdb"
con.execute(f"ATTACH '{wm_path}' AS wm (READ_ONLY)")
con.sql("SHOW TABLES FROM wm").show()


┌───────────────────┐
│       name        │
│      varchar      │
├───────────────────┤
│ bronze_manifest   │
│ watermark_history │
│ watermarks        │
└───────────────────┘



In [4]:
q("SELECT * FROM wm.watermarks ORDER BY source")


,source,dataset,cursor,rows_loaded,batches,last_load_ts,note,updated_at
0,fars,2019,"{""bytes"":26974033,""current_load_ts"":""20260907T142904680Z"",""etag"":""\""d175abc9...",1420858,1,20260907T142904680Z,full refresh (initial),2026-09-07 14:29:13.266464
1,fars,2020,"{""bytes"":31016385,""current_load_ts"":""20260907T142919259Z"",""etag"":""\""a11b0d1a...",1809904,1,20260907T142919259Z,full refresh (initial),2026-09-07 14:29:29.335634
2,fars,2021,"{""bytes"":35190858,""current_load_ts"":""20260907T142937197Z"",""etag"":""\""045ff1f1...",2025081,1,20260907T142937197Z,full refresh (initial),2026-09-07 14:29:48.445924
3,fars,2022,"{""bytes"":34689724,""current_load_ts"":""20260907T142956110Z"",""etag"":""\""4f0eb7a3...",2005295,1,20260907T142956110Z,full refresh (initial),2026-09-07 14:30:07.057269
4,fars,2023,"{""bytes"":34174899,""current_load_ts"":""20260907T143016416Z"",""etag"":""\""4f56c9e3...",1937154,1,20260907T143016416Z,full refresh (initial),2026-09-07 14:30:27.634262
5,fars,2024,"{""bytes"":32672161,""current_load_ts"":""20260907T143037417Z"",""etag"":""\""8fef3df7...",1852999,1,20260907T143037417Z,full refresh (initial),2026-09-07 14:30:47.782991
6,montgomery,bhju-22kf,"{""id"":""row-q49u-4dz6.gzee"",""updated_at"":""2026-09-04T05:42:06.051""}",125005,26,20260907T142937665Z,page 26,2026-09-07 14:30:35.941679
7,montgomery,mmzv-x632,"{""id"":""row-khvf.dhde~edk6"",""updated_at"":""2026-09-04T05:46:41.771""}",220043,45,20260907T142937665Z,page 45,2026-09-07 14:32:19.561873
8,montgomery,n7fk-dce5,"{""id"":""row-jpy2_g9mm-7dnj"",""updated_at"":""2026-09-04T05:30:40.738""}",7521,2,20260907T142937665Z,page 2,2026-09-07 14:32:24.889578
9,txdot,cris_crash,"{""bounded"":true,""last_objectid"":100000,""max_pages"":50,""oid_ceiling"":3088450,...",100000,50,20260907T142714078Z,page 50,2026-09-07 14:28:54.919273


In [5]:
q("""
SELECT source, COUNT(*) AS advances, min(advanced_at) AS first, max(advanced_at) AS last
FROM wm.watermark_history
GROUP BY 1 ORDER BY 1
""")


,source,advances,first,last
0,fars,6,2026-09-07 14:29:13.266464,2026-09-07 14:30:47.782991
1,montgomery,79,2026-09-07 14:26:59.169221,2026-09-07 14:32:24.889578
2,txdot,50,2026-09-07 14:27:15.587575,2026-09-07 14:28:54.919273


In [6]:
q("""
SELECT source, dataset, kind, COUNT(*) AS artifacts,
       sum(bytes) AS total_bytes, sum(row_count) AS total_rows
FROM wm.bronze_manifest
GROUP BY 1, 2, 3 ORDER BY 1, 2, 3
""")


,source,dataset,kind,artifacts,total_bytes,total_rows
0,fars,2019,parsed,27,17448944.0,1420858.0
1,fars,2019,raw,1,26974033.0,NaN
2,fars,2020,parsed,33,20259923.0,1809904.0
3,fars,2020,raw,1,31016385.0,NaN
4,fars,2021,parsed,33,22644186.0,2025081.0
5,fars,2021,raw,1,35190858.0,NaN
6,fars,2022,parsed,33,22564706.0,2005295.0
7,fars,2022,raw,1,34689724.0,NaN
8,fars,2023,parsed,33,22224261.0,1937154.0
9,fars,2023,raw,1,34174899.0,NaN


## 3. Bronze — Montgomery County

SoQL / Socrata. Three datasets at three grains, keyset-paginated on `(:updated_at, :id)`. Raw JSON is preserved page-for-page alongside a parsed parquet twin.

In [7]:
for ds, desc in MONTGOMERY_DATASETS.items():
    base = BRONZE_DIR / "montgomery" / ds
    parts = partitions(base)
    print(f"{ds:12s} {desc}")
    for p in parts:
        n_pages = len(list(p.glob("page_*.parquet")))
        print(f"    {p.name}  ({n_pages} pages)")


bhju-22kf    incidents (crash grain)
    20260907T142655337Z  (2 pages)
    20260907T142937665Z  (26 pages)
mmzv-x632    drivers (party grain, denormalized crash attrs)
    20260907T142655337Z  (3 pages)
    20260907T142937665Z  (45 pages)
n7fk-dce5    non_motorists (party grain)
    20260907T142655337Z  (1 pages)
    20260907T142937665Z  (2 pages)


In [8]:
# Schema + row count, LATEST partition only, per dataset
for ds in MONTGOMERY_DATASETS:
    lp = latest_partition(BRONZE_DIR / "montgomery" / ds)
    if lp is None:
        continue
    glob = str(lp / "page_*.parquet")
    n = q(f"SELECT COUNT(*) AS n FROM read_parquet('{glob}')")["n"][0]
    print(f"--- {ds}  latest={lp.name}  rows={n} ---")
display(schema(str(latest_partition(BRONZE_DIR / 'montgomery' / 'bhju-22kf') / 'page_*.parquet')))


--- bhju-22kf  latest=20260907T142937665Z  rows=125005 ---
--- mmzv-x632  latest=20260907T142937665Z  rows=220043 ---
--- n7fk-dce5  latest=20260907T142937665Z  rows=7521 ---


,column_name,column_type,null,key,default,extra
0,report_number,VARCHAR,YES,None,None,None
1,local_case_number,VARCHAR,YES,None,None,None
2,agency_name,VARCHAR,YES,None,None,None
3,acrs_report_type,VARCHAR,YES,None,None,None
4,crash_date_time,VARCHAR,YES,None,None,None
5,hit_run,VARCHAR,YES,None,None,None
6,route_type,VARCHAR,YES,None,None,None
7,lane_direction,VARCHAR,YES,None,None,None
8,number_of_lanes,VARCHAR,YES,None,None,None
9,direction,VARCHAR,YES,None,None,None


In [9]:
incidents_glob = str(latest_partition(BRONZE_DIR / "montgomery" / "bhju-22kf") / "page_*.parquet")
q(f"SELECT * FROM read_parquet('{incidents_glob}') LIMIT 5")


,report_number,local_case_number,agency_name,acrs_report_type,crash_date_time,hit_run,route_type,lane_direction,number_of_lanes,direction,distance,distance_unit,road_grade,road_name,cross_street_name,municipality,at_fault,collision_type,weather,surface_condition,light,traffic_control,driver_substance_abuse,first_harmful_event,second_harmful_event,junction,intersection_type,road_alignment,road_condition,road_division,latitude,longitude,geolocation,:id,:updated_at,:created_at,:version,_bronze_source,_bronze_dataset,_bronze_load_ts,_bronze_page,_bronze_raw_path,_bronze_row_sha256,related_non_motorist,non_motorist_substance_abuse,lane_type
0,MCP3279002W,230065167,Montgomery County Police,Property Damage Crash,2023-11-08T14:50:00.000,No,County,East,3,East,0,FEET,LEVEL,RANDOLPH RD,GLENALLAN AVE,N/A,DRIVER,STRAIGHT MOVEMENT ANGLE,N/A,DRY,DAYLIGHT,TRAFFIC SIGNAL,"N/A, NONE DETECTED",OTHER VEHICLE,N/A,INTERSECTION,FOUR-WAY INTERSECTION,STRAIGHT,NO DEFECTS,"TWO-WAY, DIVIDED, POSITIVE MEDIAN BARRIER",39.06048814,-77.04483823,"{""human_address"":""{\""address\"": \""\"", \""city\"": \""\"", \""state\"": \""\"", \""zip...",row-usr2.75aj.3qmk,2024-06-12T20:28:27.326Z,2024-06-12T20:27:58.211Z,rv-wkma~t85h-ki8g,montgomery,bhju-22kf,20260907T142937665Z,00001,page_00001.json,6d67feb05eb383c72e96c49cf9c8c72fdf5a92a3da57231b4dea3fb0e504b98e,None,None,None
1,MCP2954003H,170516259,Montgomery County Police,Injury Crash,2017-06-28T16:51:00.000,No,Maryland (State),West,3,East,30,FEET,LEVEL,UNIVERSITY BLVD E,SEPTEMBER LA,N/A,DRIVER,SAME DIR REAR END,CLEAR,DRY,DAYLIGHT,TRAFFIC SIGNAL,NONE DETECTED,OTHER VEHICLE,N/A,N/A,N/A,STRAIGHT,NO DEFECTS,"TWO-WAY, DIVIDED, POSITIVE MEDIAN BARRIER",39.00410296,-76.99654341,"{""latitude"":""39.00410296"",""longitude"":""-76.99654341""}",row-ce3q~j26p-mv4c,2024-06-12T20:28:27.326Z,2024-06-12T20:27:58.211Z,rv-e6wy-dms2.nfs3,montgomery,bhju-22kf,20260907T142937665Z,00001,page_00001.json,71fb1b77797701881cf8b44b946b67f4af7c2acdbb963f4e7661e51294de1568,None,None,None
2,MCP3123005Y,230033149,Montgomery County Police,Property Damage Crash,2023-07-12T08:07:00.000,No,Maryland (State),West,2,East,0,FEET,GRADE DOWNHILL,MUNCASTER MILL RD,BOWIE MILL RD,N/A,DRIVER,SAME DIR REAR END,CLEAR,DRY,DAYLIGHT,TRAFFIC SIGNAL,N/A,OTHER VEHICLE,N/A,INTERSECTION,T-INTERSECTION,STRAIGHT,NO DEFECTS,"TWO-WAY, NOT DIVIDED",39.13377798,-77.12233117,"{""human_address"":""{\""address\"": \""\"", \""city\"": \""\"", \""state\"": \""\"", \""zip...",row-cs5z_bqgp-b3nq,2024-06-12T20:28:27.326Z,2024-06-12T20:27:58.211Z,rv-wdsn.g5w7~9vky,montgomery,bhju-22kf,20260907T142937665Z,00001,page_00001.json,20e6187135457c808c8462a9b16fdc80e58215b41d1bfcd5544adfa061b4494e,None,None,None
3,MCP31480051,230030803,Montgomery County Police,Property Damage Crash,2023-06-27T18:59:00.000,No,County,North,1,South,0,FEET,LEVEL,MANOR RD,VILLAGE PARK PL,N/A,DRIVER,OTHER,RAINING,WET,DAYLIGHT,N/A,NONE DETECTED,PARKED VEHICLE,N/A,INTERSECTION,T-INTERSECTION,STRAIGHT,NO DEFECTS,"TWO-WAY, DIVIDED, UNPROTECTED PAINTED MIN 4 FEET",38.996545,-77.07497667,"{""human_address"":""{\""address\"": \""\"", \""city\"": \""\"", \""state\"": \""\"", \""zip...",row-z7jq_p6zw.z79v,2024-06-12T20:28:27.326Z,2024-06-12T20:27:58.211Z,rv-enjf.u3y5-skv5,montgomery,bhju-22kf,20260907T142937665Z,00001,page_00001.json,aad763220ddded7f929c5162b006357d256fb4fc82fbf290648a6a012e949f57,None,None,None
4,MCP3093001C,180056633,Montgomery County Police,Injury Crash,2018-11-11T18:13:00.000,No,Maryland (State),North,1,North,0,FEET,GRADE DOWNHILL,BROOKVILLE RD,EAST WEST HWY,N/A,DRIVER,SAME DIR REAR END,N/A,DRY,DARK LIGHTS ON,STOP SIGN,N/A,OTHER VEHICLE,N/A,INTERSECTION,T-INTERSECTION,STRAIGHT,NO DEFECTS,ONE-WAY TRAFFICWAY,38.98927333,-77.07020167,"{""latitude"":""38.98927333"",""longitude"":""-77.07020167""}",row-3t7f.ffzc.tb7s,2024-06-12T20:28:27.326Z,2024-06-12T20:27:58.211Z,rv-xeqc.2due_iutx,montgomery,bhju-22kf,20260907T142937665Z,00001,page_00001.json,bf42f99f6ce8816f747e1e306d1f4a597158de5a1d606445dab14a585f8b1

In [10]:
drivers_glob = str(latest_partition(BRONZE_DIR / "montgomery" / "mmzv-x632") / "page_*.parquet")
q(f"SELECT * FROM read_parquet('{drivers_glob}') LIMIT 5")


,report_number,local_case_number,agency_name,acrs_report_type,crash_date_time,route_type,road_name,cross_street_name,municipality,collision_type,weather,surface_condition,light,traffic_control,driver_substance_abuse,person_id,driver_at_fault,injury_severity,circumstance,driver_distracted_by,drivers_license_state,vehicle_id,vehicle_damage_extent,vehicle_first_impact_location,vehicle_body_type,vehicle_movement,vehicle_going_dir,speed_limit,driverless_vehicle,parked_vehicle,vehicle_year,vehicle_make,vehicle_model,latitude,longitude,geolocation,:id,:updated_at,:created_at,:version,_bronze_source,_bronze_dataset,_bronze_load_ts,_bronze_page,_bronze_raw_path,_bronze_row_sha256,off_road_description,related_non_motorist,non_motorist_substance_abuse
0,MCP12270021,230063063,Montgomery County Police,Fatal Crash,2023-10-28T12:15:00.000,Ramp,RAMP 8 FR US 29 SB TO DUSTIN RD,DUSTIN RD,N/A,SINGLE VEHICLE,CLEAR,DRY,DAYLIGHT,YIELD SIGN,NONE DETECTED,287FDF5B-58BB-4BED-BD4A-E4FE79661841,Yes,FATAL INJURY,N/A,UNKNOWN,MD,A1186BA2-9E54-4516-B100-F00DBE847FD5,DESTROYED,TWELVE OCLOCK,PASSENGER CAR,MOVING CONSTANT SPEED,North,30,No,No,2009,HYUNDAI,ELANTRA,39.12246766,-76.92633791,"{""human_address"":""{\""address\"": \""\"", \""city\"": \""\"", \""state\"": \""\"", \""zip...",row-5zkw-6jp9_xmvd,2024-06-12T15:04:17.445Z,2024-06-12T15:02:48.323Z,rv-hueg.tcmw~jhjd,montgomery,mmzv-x632,20260907T142937665Z,00001,page_00001.json,d335bcd976710f4b9be6e24f2156eb112227ddbbb7f546d5bb11dfc780af258b,None,None,None
1,EJ7851009P,230017554,Gaithersburg Police Depar,Injury Crash,2023-04-13T01:07:00.000,Maryland (State),S FREDERICK RD,EDUCATION BLVD,GAITHERSBURG,OTHER,CLEAR,DRY,DARK LIGHTS ON,NO CONTROLS,UNKNOWN,E65D73D1-1223-410C-ABE9-2228F8B35627,Yes,NO APPARENT INJURY,N/A,UNKNOWN,NaN,F5CD11CF-A001-4B1B-B482-C8A5FEF0A0FA,NO DAMAGE,NON-COLLISION,NaN,CHANGING LANES,North,30,No,No,0,UNKNOWN,UNKNOWN,39.13646062,-77.19249218,"{""human_address"":""{\""address\"": \""\"", \""city\"": \""\"", \""state\"": \""\"", \""zip...",row-8ekk~2pvr_9799,2024-06-12T15:04:17.445Z,2024-06-12T15:02:48.323Z,rv-5i3d_yj49~nb46,montgomery,mmzv-x632,20260907T142937665Z,00001,page_00001.json,ba12652ffbba8157d2a8d066ecd66861138f325b3ea0a951ad9959b10e67500e,None,None,None
2,MCP3129008C,230074266,Montgomery County Police,Property Damage Crash,2023-12-30T11:21:00.000,Maryland (State),CONNECTICUT AVE,WELLER RD,N/A,STRAIGHT MOVEMENT ANGLE,CLOUDY,DRY,DAYLIGHT,TRAFFIC SIGNAL,NONE DETECTED,AAA2C6D2-5B2C-4D9E-84B0-479AA6E6522C,No,NO APPARENT INJURY,N/A,LOOKED BUT DID NOT SEE,MD,5CDD6C3B-EBDD-4505-8E86-9BBDE567696B,FUNCTIONAL,TWELVE OCLOCK,PASSENGER CAR,MOVING CONSTANT SPEED,South,35,No,No,2003,HOND,4S,39.063112,-77.07361733,"{""human_address"":""{\""address\"": \""\"", \""city\"": \""\"", \""state\"": \""\"", \""zip...",row-mamb~5xq6~sbij,2024-06-12T15:04:17.445Z,2024-06-12T15:02:48.323Z,rv-vyyc-fknr-prbb,montgomery,mmzv-x632,20260907T142937665Z,00001,page_00001.json,9076bcba0ca63a42584f9e98e707b219e654d9776568b6d35fb7eacde6fcafc0,None,None,None
3,MCP27520061,230071949,Montgomery County Police,Property Damage Crash,2023-12-15T16:30:00.000,Maryland (State),WILSON LA,BRAEBURN PL,N/A,SAME DIR REAR END,CLEAR,DRY,DAYLIGHT,NO CONTROLS,NONE DETECTED,8326DF89-A4CA-46C1-BFF8-FB4C2DE76BD1,No,NO APPARENT INJURY,N/A,NOT DISTRACTED,VA,9510AAB7-EBB6-4E79-BA7A-8384C1C0720A,NO DAMAGE,SIX OCLOCK,PICKUP TRUCK,SLOWING OR STOPPING,West,30,No,No,2018,CHEVY,SILVERADO,38.97408079,-77.14531362,"{""human_address"":""{\""address\"": \""\"", \""city\"": \""\"", \""state\"": \""\"", \""zip...",row-nahp.wrns-iik3,2024-06-12T15:04:17.445Z,2024-06-12T15:02:48.323Z,rv-wexq-a6mh_g2ze,montgomery,mmzv-x632,20260907T142937665Z,00001,page_00001.json,48b72cbe09c85437d1441a2648bcc591cdc6de50dbf7e6ca9979c2bcdd37c2f9,None,None,None
4,MCP3133004Z,230065159,Montgomery County Police,Property Damage Crash,2023-11-08T14:28:00.000,Maryland (State),NORBECK RD,NORBECK BLVD,N/A,SAME DIR REAR END,CLEAR,DRY,DAYLIGHT,TRAFFIC SIGNAL,NONE DETECTED,3B165B3B-887

In [11]:
non_motorists_glob = str(latest_partition(BRONZE_DIR / "montgomery" / "n7fk-dce5") / "page_*.parquet")
q(f"SELECT * FROM read_parquet('{non_motorists_glob}') LIMIT 5")


,report_number,local_case_number,agency_name,acrs_report_type,crash_date_time,route_type,road_name,cross_street_name,municipality,related_non_motorist,collision_type,weather,surface_condition,light,traffic_control,driver_substance_abuse,non_motorist_substance_abuse,person_id,pedestrian_type,pedestrian_movement,pedestrian_actions,pedestrian_location,at_fault,injury_severity,safety_equipment,latitude,longitude,geolocation,:id,:updated_at,:created_at,:version,_bronze_source,_bronze_dataset,_bronze_load_ts,_bronze_page,_bronze_raw_path,_bronze_row_sha256,off_road_description
0,MCP2536004X,230074244,Montgomery County Police,Injury Crash,2023-12-30T04:55:00.000,Maryland (State),GEORGIA AVE,RANDOLPH RD,N/A,OTHER,OTHER,CLEAR,DRY,DARK LIGHTS ON,NO CONTROLS,ALCOHOL PRESENT,NONE DETECTED,A84A974D-89AF-44DC-A7EA-26FA8BD9748F,OTHER,Other Working,NO IMPROPER ACTIONS,SIDEWALK,No,SUSPECTED SERIOUS INJURY,N/A,39.05920152,-77.05011059,"{""human_address"":""{\""address\"": \""\"", \""city\"": \""\"", \""state\"": \""\"", \""zip...",row-xepx-xkes_ym3n,2024-06-12T15:17:29.770Z,2024-06-12T15:17:26.302Z,rv-63kr-inuk~b6hn,montgomery,n7fk-dce5,20260907T142937665Z,00001,page_00001.json,68d6575130f855f1c1fa345e029d1e38ed74e569337d0e5a517fb7b2eb4e9c9a,NaN
1,MCP3254003N,240000590,Montgomery County Police,Injury Crash,2023-12-31T13:00:00.000,NaN,NaN,NaN,NaN,PEDESTRIAN,SINGLE VEHICLE,CLEAR,NaN,DAYLIGHT,TRAFFIC SIGNAL,UNKNOWN,N/A,1BA94396-6FC0-4510-8C88-EA1FEC1BDD8A,PEDESTRIAN,Walking/Cycling on Sidewalk,NO IMPROPER ACTIONS,SIDEWALK,No,SUSPECTED MINOR INJURY,N/A,39.14832345,-77.20689024,"{""human_address"":""{\""address\"": \""\"", \""city\"": \""\"", \""state\"": \""\"", \""zip...",row-bet4~d43i-bux8,2024-06-12T15:17:29.770Z,2024-06-12T15:17:26.302Z,rv-4dpa-nqp4.dj2n,montgomery,n7fk-dce5,20260907T142937665Z,00001,page_00001.json,0730cf778d5a715c30606eb8e901db49f1e63633411ab6532d45498c0080b3d6,ON THE SIDEWALK IN FRONT OF EXXON AT 448 N FREDERICK AVENUE GAITHERSBURG MD ...
2,MCP3039006P,230072243,Montgomery County Police,Injury Crash,2023-12-17T17:43:00.000,Maryland (State),GEORGIA AVE,COLESVILLE RD,N/A,PEDESTRIAN,STRAIGHT MOVEMENT ANGLE,RAINING,WET,DARK LIGHTS ON,TRAFFIC SIGNAL,NONE DETECTED,NONE DETECTED,6900AAFD-0EBB-4A36-B8D4-9B25E066A795,PEDESTRIAN,Cross/Enter at Intersection,NO IMPROPER ACTIONS,ON ROADWAY AT CROSSWALK,No,POSSIBLE INJURY,N/A,38.99589921,-77.02818753,"{""human_address"":""{\""address\"": \""\"", \""city\"": \""\"", \""state\"": \""\"", \""zip...",row-vkp7~77ux.g6br,2024-06-12T15:17:29.770Z,2024-06-12T15:17:26.302Z,rv-zc6u-mtet_wfzw,montgomery,n7fk-dce5,20260907T142937665Z,00001,page_00001.json,96443c11ca42737208d4db66e49a896d8a5f26593639093e74d80d7f33da7792,NaN
3,MCP3254003K,230072050,Montgomery County Police,Injury Crash,2023-12-16T12:36:00.000,Maryland (State),GERMANTOWN RD,MIDDLEBROOK RD,N/A,BICYCLIST,STRAIGHT MOVEMENT ANGLE,CLEAR,DRY,DAYLIGHT,TRAFFIC SIGNAL,NONE DETECTED,NONE DETECTED,0555B6BA-0A80-4556-97C8-DD2FC0027B91,BICYCLIST,Cross/Enter at Intersection,FAILURE TO OBEY TRAFFIC SIGNS SIGNALS OR OFFICER,ON ROADWAY AT CROSSWALK,Yes,SUSPECTED MINOR INJURY,MC/BIKE HELMET,39.17877577,-77.26718974,"{""human_address"":""{\""address\"": \""\"", \""city\"": \""\"", \""state\"": \""\"", \""zip...",row-va9y_gmpk_87r9,2024-06-12T15:17:29.770Z,2024-06-12T15:17:26.302Z,rv-x2jb~a252~z2u4,montgomery,n7fk-dce5,20260907T142937665Z,00001,page_00001.json,8a6b32dce6895d4cb24fa704b65792241db331647d988dbe783dca1a34414162,NaN
4,MCP2563001L,230017107,Montgomery County Police,Injury Crash,2023-04-10T16:21:00.000,Maryland (State),MUNCASTER MILL RD,STRUC #15015 ROCK CREEK,N/A,PEDESTRIAN,SINGLE VEHICLE,CLEAR,DRY,DAYLIGHT,WARNING SIGN,NONE DETECTED,NONE DETECTED,2D1655C2-78EB-4552-899C-4C57B12B59E9,PEDESTRIAN,Cross/Enter not at Intersection,FAILURE TO YIELD RIGHT OF WAY,ON ROADWAY AT CROSSWALK,Yes,SUSPECTED SERIOUS INJURY,NONE,39.13764511,-77.12946041,"{""human_address"":""{\""address\"": \""\"", \""city\"": \""\"", \""state\"": \""\"", \""zip...",row-8fr7~em5

### 3a. Known defect — out-of-envelope coordinates (`bhju-22kf`)

Zero nulls, zero zero-valued lat/long, and still wrong: some rows sit well outside Montgomery County. `DATA_QUALITY.md` reports 114 rows outside the assignment's stated bbox on the full history; this is the same check against the latest bronze partition.

In [12]:
q(f"""
SELECT COUNT(*) AS n_rows,
       COUNT(*) FILTER (WHERE latitude IS NULL OR longitude IS NULL) AS n_null,
       COUNT(*) FILTER (WHERE TRY_CAST(latitude AS DOUBLE) = 0 OR TRY_CAST(longitude AS DOUBLE) = 0) AS n_zero,
       COUNT(*) FILTER (
           WHERE TRY_CAST(latitude AS DOUBLE) NOT BETWEEN 38.9 AND 39.36
              OR TRY_CAST(longitude AS DOUBLE) NOT BETWEEN -77.54 AND -76.87
       ) AS n_outside_stated_bbox
FROM read_parquet('{incidents_glob}')
""")


,n_rows,n_null,n_zero,n_outside_stated_bbox
0,125005,0,0,114


In [13]:
# The furthest offenders
q(f"""
SELECT report_number, latitude, longitude
FROM read_parquet('{incidents_glob}')
WHERE TRY_CAST(latitude AS DOUBLE) NOT BETWEEN 38.9 AND 39.36
   OR TRY_CAST(longitude AS DOUBLE) NOT BETWEEN -77.54 AND -76.87
ORDER BY abs(TRY_CAST(latitude AS DOUBLE) - 39.13) + abs(TRY_CAST(longitude AS DOUBLE) + 77.2) DESC
LIMIT 10
""")


,report_number,latitude,longitude
0,MCP1291002F,37.72,-79.48
1,MCP12910029,39.72,-79.486
2,MCP12910025,39.72,-79.486
3,MCP1291002C,39.72,-79.486
4,MCP12910023,39.72,-79.486
5,MCP12910026,39.72,-79.486
6,MCP12910028,39.72,-79.486
7,MCP12910027,39.72,-79.486
8,MCP23560009,38.77121637,-79.42565918
9,DM83880030,38.55400492,-79.18192616


### 3b. Known defect — two dictionary generations concatenated (`mmzv-x632.driver_substance_abuse`)

Old scheme: uppercase, single value (`NONE DETECTED`). New scheme: title-case, comma-joined pair (`Not Suspect of Alcohol Use, Not Suspect of Drug Use`). Both are live in the same column.

In [14]:
q(f"""
SELECT driver_substance_abuse, COUNT(*) AS n
FROM read_parquet('{drivers_glob}')
GROUP BY 1
ORDER BY n DESC
""")


,driver_substance_abuse,n
0,NONE DETECTED,122544
1,"Not Suspect of Alcohol Use, Not Suspect of Drug Use",40832
2,N/A,31320
3,UNKNOWN,11990
4,"Unknown, Unknown",5462
5,ALCOHOL PRESENT,4087
6,ALCOHOL CONTRIBUTED,1435
7,"Suspect of Alcohol Use, Not Suspect of Drug Use",1151
8,ILLEGAL DRUG PRESENT,259
9,"Suspect of Alcohol Use, Unknown",143


### 3c. Known defect — comma-joined `number_of_lanes` (Incidents)

In [15]:
q(f"""
SELECT number_of_lanes, COUNT(*) AS n
FROM read_parquet('{incidents_glob}')
WHERE number_of_lanes LIKE '%,%'
GROUP BY 1 ORDER BY n DESC
""")


,number_of_lanes,n
0,"2, 3",589
1,"1, 2",582
2,"3, 4",386
3,"1, 3",360
4,"2, 4",253
...,...,...
91,"0, 4, 5",1
92,"5, 8",1
93,"3, 93",1
94,"10, 3",1


### 3d. Known defect — the tables disagree about the crash universe

Anti-join `report_number` between Incidents and Drivers, both directions.

In [16]:
q(f"""
WITH inc AS (SELECT DISTINCT report_number FROM read_parquet('{incidents_glob}')),
     drv AS (SELECT DISTINCT report_number FROM read_parquet('{drivers_glob}'))
SELECT
  (SELECT COUNT(*) FROM inc WHERE report_number NOT IN (SELECT report_number FROM drv)) AS incidents_without_a_driver,
  (SELECT COUNT(*) FROM drv WHERE report_number NOT IN (SELECT report_number FROM inc)) AS drivers_without_an_incident
""")


,incidents_without_a_driver,drivers_without_an_incident
0,785,0


### 3e. Known defect — grain fan-out

Drivers is one row per driver but denormalizes crash-level fields. Rows per `report_number` on the latest partition:

In [17]:
q(f"""
SELECT COUNT(*)::DOUBLE / COUNT(DISTINCT report_number) AS rows_per_crash
FROM read_parquet('{drivers_glob}')
""")


,rows_per_crash
0,1.771398


## 4. Bronze — TxDOT CRIS

ArcGIS FeatureServer, `ESRI_OID`-keyset paginated, 2,000 rows/page, bounded to the first 100,000 OIDs locally (`--full` removes the bound). Raw pages are gzipped.

In [ ]:
txdot_base = BRONZE_DIR / "txdot" / "cris_crash"
for p in partitions(txdot_base):
    n_pages = len(list(p.glob("page_*.parquet")))
    print(f"{p.name}  ({n_pages} pages)")

txdot_glob = str(latest_partition(txdot_base) / "page_*.parquet")
schema(txdot_glob)


In [ ]:
q(f"SELECT COUNT(*) AS n, COUNT(DISTINCT crash_id) AS distinct_crash_id FROM read_parquet('{txdot_glob}')")


In [ ]:
q(f"SELECT * FROM read_parquet('{txdot_glob}') LIMIT 5")


### 4a. Amended reports (`amend_supp_fl`) — restatement is not an edge case

In [ ]:
q(f"""
SELECT amend_supp_fl, COUNT(*) AS n, round(100.0 * COUNT(*) / sum(COUNT(*)) OVER (), 1) AS pct
FROM read_parquet('{txdot_glob}')
GROUP BY 1 ORDER BY 1
""")


### 4b. Two competing coordinate pairs

`located_fl` should be exactly "CRIS produced a derived location".

In [ ]:
q(f"""
SELECT
  located_fl,
  (latitude IS NOT NULL AND longitude IS NOT NULL)         AS derived_present,
  (rpt_latitude IS NOT NULL AND rpt_longitude IS NOT NULL) AS officer_present,
  COUNT(*) AS n
FROM read_parquet('{txdot_glob}')
GROUP BY 1, 2, 3
ORDER BY n DESC
""")


### 4c. `crash_sev_id` — opaque code, verified against injury counts

In [ ]:
# All TxDOT bronze columns are VARCHAR (esri fields are typed as strings), hence the TRY_CASTs.
q(f"""
SELECT TRY_CAST(crash_sev_id AS INTEGER) AS crash_sev_id, COUNT(*) AS n,
       sum(CASE WHEN TRY_CAST(death_cnt AS DOUBLE) > 0 THEN 1 ELSE 0 END) AS with_fatal,
       sum(CASE WHEN TRY_CAST(sus_serious_injry_cnt AS DOUBLE) > 0 THEN 1 ELSE 0 END) AS with_serious,
       sum(CASE WHEN TRY_CAST(non_injry_cnt AS DOUBLE) > 0 THEN 1 ELSE 0 END) AS with_non_injury
FROM read_parquet('{txdot_glob}')
GROUP BY 1 ORDER BY 1
""")


## 5. Bronze — FARS

One zip per year, full-refresh-with-restatement. Each year's partition carries ~27 member tables; `accident`, `vehicle`, `person` are the ones the pipeline transforms today.

In [ ]:
fars_base = BRONZE_DIR / "fars"
fars_years = sorted(p.name for p in fars_base.iterdir() if p.is_dir())
print("years on disk:", fars_years)

for y in fars_years:
    lp = latest_partition(fars_base / y)
    tables = sorted(p.stem for p in lp.glob("*.parquet"))
    print(f"{y}  {lp.name}  {len(tables)} tables")


In [ ]:
# Row counts per year for the three tables the pipeline actually uses
rows = []
for y in fars_years:
    lp = latest_partition(fars_base / y)
    for tbl in ("accident", "vehicle", "person"):
        f = lp / f"{tbl}.parquet"
        if f.exists():
            n = q(f"SELECT COUNT(*) AS n FROM read_parquet('{f}')")["n"][0]
            rows.append({"year": y, "table": tbl, "rows": n})
pd.DataFrame(rows).pivot(index="year", columns="table", values="rows")


In [ ]:
accident_glob = str(fars_base / "*" / "*" / "accident.parquet")
schema(accident_glob)


In [ ]:
q(f"SELECT * FROM read_parquet('{accident_glob}', union_by_name=True) LIMIT 5")


### 5a. Known defect — sentinel coordinates, not nulls

`LATITUDE` sentinels are `77.7777` / `88.8888` / `99.9999`; `LONGITUD` sentinels are the *three-digit* mirror `777.7777` / `888.8888` / `999.9999`. Formatting drifts between years (`77.7777` vs `77.77770000`), so matching must be numeric, never a string compare.

In [ ]:
q(f"""
SELECT _bronze_dataset AS year,
       COUNT(*) FILTER (
         WHERE abs(TRY_CAST(LATITUDE AS DOUBLE) - 77.7777) < 1e-4
            OR abs(TRY_CAST(LATITUDE AS DOUBLE) - 88.8888) < 1e-4
            OR abs(TRY_CAST(LATITUDE AS DOUBLE) - 99.9999) < 1e-4
       ) AS sentinel_lat_numeric,
       COUNT(*) FILTER (WHERE LATITUDE IN ('77.7777','88.8888','99.9999')) AS sentinel_lat_exact_string
FROM read_parquet('{accident_glob}', union_by_name=True)
GROUP BY 1 ORDER BY 1
""")


In [ ]:
# HOUR = 99 ('unknown') per year
q(f"""
SELECT _bronze_dataset AS year, COUNT(*) FILTER (WHERE TRY_CAST(HOUR AS INTEGER) = 99) AS hour_unknown,
       COUNT(*) AS total
FROM read_parquet('{accident_glob}', union_by_name=True)
GROUP BY 1 ORDER BY 1
""")


## 6. Silver — per-source current/history tables

SCD2 output of `python -m src.transform.build`. `_current` is the latest version per natural key; `_history` keeps every version.

In [ ]:
for f in sorted(SILVER_DIR.rglob("*.parquet")):
    n = q(f"SELECT COUNT(*) AS n FROM read_parquet('{f}')")["n"][0]
    print(f"{f.relative_to(SILVER_DIR)!s:45s} {n:>10,d} rows")


### 6a. Silver — Montgomery

In [ ]:
moco_crash = SILVER_DIR / "montgomery" / "crash_current.parquet"
schema(str(moco_crash))


In [ ]:
q(f"SELECT * FROM read_parquet('{moco_crash}') LIMIT 5")


In [ ]:
# geo_quality resolution of the out-of-envelope defect
q(f"SELECT geo_quality, COUNT(*) AS n FROM read_parquet('{moco_crash}') GROUP BY 1 ORDER BY n DESC")


In [ ]:
q(f"SELECT has_driver_rows, COUNT(*) AS n FROM read_parquet('{moco_crash}') GROUP BY 1")


In [ ]:
q(f"SELECT severity_grain, COUNT(*) AS n FROM read_parquet('{moco_crash}') GROUP BY 1 ORDER BY n DESC")


In [ ]:
moco_driver = SILVER_DIR / "montgomery" / "driver_current.parquet"
q(f"""
SELECT substance_scheme, alcohol_status, drug_status, COUNT(*) AS n
FROM read_parquet('{moco_driver}')
GROUP BY 1, 2, 3 ORDER BY n DESC
""")


In [ ]:
moco_nm = SILVER_DIR / "montgomery" / "non_motorist_current.parquet"
q(f"SELECT COUNT(*) AS n, COUNT(*) FILTER (WHERE duplicate_person_id) AS n_dup FROM read_parquet('{moco_nm}')")


### 6b. Silver — TxDOT

In [ ]:
txd_crash = SILVER_DIR / "txdot" / "crash_current.parquet"
schema(str(txd_crash))


In [ ]:
q(f"SELECT * FROM read_parquet('{txd_crash}') LIMIT 5")


In [ ]:
q(f"SELECT coord_source, COUNT(*) AS n FROM read_parquet('{txd_crash}') GROUP BY 1 ORDER BY n DESC")


In [ ]:
txd_history = SILVER_DIR / "txdot" / "crash_history.parquet"
q(f"""
SELECT (SELECT COUNT(*) FROM read_parquet('{txd_crash}'))   AS current_rows,
       (SELECT COUNT(*) FROM read_parquet('{txd_history}')) AS history_rows,
       (SELECT COUNT(*) FILTER (WHERE is_amended) FROM read_parquet('{txd_crash}')) AS amended_current
""")


### 6c. Silver — FARS

In [ ]:
fars_accident = SILVER_DIR / "fars" / "accident_current.parquet"
schema(str(fars_accident))


In [ ]:
q(f"SELECT * FROM read_parquet('{fars_accident}') LIMIT 5")


In [ ]:
fars_codebook = SILVER_DIR / "fars" / "codebook.parquet"
q(f"SELECT COUNT(*) AS n FROM read_parquet('{fars_codebook}')")


In [ ]:
q(f"SELECT * FROM read_parquet('{fars_codebook}') LIMIT 10")


## 7. Silver — `crash_current` (unified, cross-source, NOT entity-resolved)

One row per crash *per source*. A Texas fatality can legitimately appear once from TxDOT and once from FARS — this table does not try to reconcile that, by design (see `src/transform/unified.py`).

In [ ]:
crash_current = SILVER_DIR / "crash_current.parquet"
schema(str(crash_current))


In [ ]:
q(f"SELECT * FROM read_parquet('{crash_current}') LIMIT 5")


In [ ]:
q(f"""
SELECT source_system, COUNT(*) AS n, COUNT(DISTINCT crash_uid) AS distinct_crash_uid,
       min(crash_date) AS min_date, max(crash_date) AS max_date
FROM read_parquet('{crash_current}')
GROUP BY 1 ORDER BY 1
""")


In [ ]:
q(f"SELECT jurisdiction, COUNT(*) AS n FROM read_parquet('{crash_current}') GROUP BY 1 ORDER BY n DESC")


In [ ]:
q(f"""
SELECT source_system, severity_ordinal, COUNT(*) AS n
FROM read_parquet('{crash_current}')
GROUP BY 1, 2 ORDER BY 1, 2
""")


In [ ]:
q(f"SELECT geo_quality, COUNT(*) AS n FROM read_parquet('{crash_current}') GROUP BY 1 ORDER BY n DESC")


## 8. Visual EDA

In [ ]:
by_source = q(f"SELECT source_system, COUNT(*) AS n FROM read_parquet('{crash_current}') GROUP BY 1 ORDER BY n DESC")
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(by_source["source_system"], by_source["n"])
ax.set_title("Crashes by source system (crash_current)")
ax.set_ylabel("rows")
plt.tight_layout()
plt.show()


In [ ]:
sev = q(f"""
SELECT source_system, severity_ordinal, COUNT(*) AS n
FROM read_parquet('{crash_current}')
WHERE severity_ordinal IS NOT NULL
GROUP BY 1, 2 ORDER BY 1, 2
""")
pivot = sev.pivot(index="severity_ordinal", columns="source_system", values="n").fillna(0)
pivot.plot(kind="bar", stacked=True, figsize=(7, 4))
plt.title("Severity ordinal distribution by source")
plt.ylabel("rows")
plt.tight_layout()
plt.show()


In [ ]:
by_month = q(f"""
SELECT source_system, date_trunc('month', crash_date) AS month, COUNT(*) AS n
FROM read_parquet('{crash_current}')
WHERE crash_date IS NOT NULL
GROUP BY 1, 2 ORDER BY 2
""")
fig, ax = plt.subplots(figsize=(9, 4))
for src, grp in by_month.groupby("source_system"):
    ax.plot(grp["month"], grp["n"], label=src)
ax.set_title("Crashes per month by source")
ax.legend()
plt.tight_layout()
plt.show()


## 9. Live defect reproduction

Quick sanity checks against the claims in `DATA_QUALITY.md` / the phase build reports, run live against whatever is on disk right now (these numbers will differ from the reports if bronze has been re-ingested since).

In [ ]:
# Dictionary cutover overlap window, on the latest Drivers bronze partition
q(f"""
WITH classified AS (
  SELECT TRY_CAST(crash_date_time AS DATE) AS crash_date,
         CASE WHEN driver_substance_abuse LIKE '%,%' THEN 'NEW_PAIR' ELSE 'OLD_SINGLE' END AS generation
  FROM read_parquet('{drivers_glob}')
  WHERE driver_substance_abuse IS NOT NULL
)
SELECT crash_date,
       COUNT(*) FILTER (WHERE generation = 'OLD_SINGLE') AS old_gen,
       COUNT(*) FILTER (WHERE generation = 'NEW_PAIR') AS new_gen
FROM classified
GROUP BY 1
HAVING COUNT(*) FILTER (WHERE generation = 'OLD_SINGLE') > 0
   AND COUNT(*) FILTER (WHERE generation = 'NEW_PAIR') > 0
ORDER BY 1
""")


In [ ]:
# TxDOT coordinate disagreement where both pairs are populated
q(f"""
SELECT COUNT(*) AS both_present,
       COUNT(*) FILTER (
         WHERE abs(TRY_CAST(latitude AS DOUBLE) - TRY_CAST(rpt_latitude AS DOUBLE)) > 0.01
            OR abs(TRY_CAST(longitude AS DOUBLE) - TRY_CAST(rpt_longitude AS DOUBLE)) > 0.01
       ) AS disagree_gt_0_01_deg
FROM read_parquet('{txdot_glob}')
WHERE latitude IS NOT NULL AND rpt_latitude IS NOT NULL
""")


## 10. Scratch space

`con` is a live DuckDB connection; `BRONZE_DIR` / `SILVER_DIR` are `pathlib.Path`s. Everything above is read-only against the parquet files, so ad hoc queries here are safe to run repeatedly.